# Parse HTML Docs
This notebook contains functions that:
1) Retrieves a list of fanwork links from a text file created with the functions in get_fanwork_links.ipynb
2) For each fanwork link, creates a BeautifulSoup object, parses the resulting html document, and extracts desired metadata about each fanwork
3) Saves the extracted metadata in a json structure

In [2]:
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup
import re
import json

# Retrieve fanworks list from text file

In [3]:
def get_fanwork_links(filepath):
    """Takes in a txt file of a pagination link and returns a list of fanwork links."""
    fanwork_links = open(filepath).readlines()
    for i in range(len(fanwork_links)):
        fanwork_links[i] = fanwork_links[i].replace("\n","")
    return fanwork_links

# Create soup objects

In [4]:
def get_fanwork_html(url):
    """
    Accesses fanwork link and retrieves outer_HTML element (which includes the HTML section that holds
    fanwork metadata).
    """
    driver = webdriver.Firefox()
    driver.get(url)

    # detect TOS agreement
    agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
    print("TOS agreement section is present: ", str(agreement_class_present))

    # if agreement class is present, find TOS agreement checkboxes / submit button
    if agreement_class_present:
        wait = WebDriverWait(driver, 10)
        tos_agree = wait.until(EC.element_to_be_clickable((By.ID, 'tos_agree')))

        # click checkboxes + accept TOS
        actions = ActionChains(driver)
        actions.move_to_element(tos_agree).perform()
        actions.click(tos_agree).perform()

        data_processing_agree = wait.until(EC.element_to_be_clickable((By.ID, 'data_processing_agree')))
        actions.move_to_element(data_processing_agree).perform()
        actions.click(data_processing_agree).perform()

        accept_tos_button = wait.until(EC.element_to_be_clickable((By.ID, 'accept_tos')))
        actions.move_to_element(accept_tos_button).perform()
        actions.click(accept_tos_button).perform()

    else:
        pass

    # detect adult content warning
    accept_adult_content_present = len(driver.find_elements(By.LINK_TEXT, 'Yes, Continue')) > 0
    print("Adult content warning is present: ", str(accept_adult_content_present))

    if accept_adult_content_present:
        wait = WebDriverWait(driver, 15)
        accept_adult_content_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Yes, Continue')))

        # click link + accept adult content warning
        actions = ActionChains(driver)
        actions.move_to_element(accept_adult_content_button).perform()
        actions.click(accept_adult_content_button).perform()

    else:
        pass

    # return page source / html doc
    driver.implicitly_wait(10) # seconds

    # get OuterHTML from 'work meta group'
    work_meta_group = driver.find_element(By.CLASS_NAME, 'wrapper')
    outer_html = work_meta_group.get_attribute('outerHTML')

    return outer_html

In [5]:
def get_soup(outer_html):
    """
    Takes in outer_HTML object from get_fanwork_html and returns BeautifulSoup object.
    """
    soup = BeautifulSoup(outer_html, features='html.parser')
    return soup

In [6]:
def detect_adult_content_warning(soup):
    """
    Reads BeautifulSoup object for a fanwork link and detects if the "adult content warning" is present.
    Works with this warning are rated either "Mature" or "Explicit" by their author.
    Works without this warning are rated either "General Audiences", "Teen And Up Audiences", or "Not Rated".
    The presence of the 'adult content warning' influences where metadata elements are located in the BeautifulSoup object, so different functions are needed to access metadata in 'adult' and 'non-adult' fanwork links.
    """
    h2 = soup.find('h2')
    string = str(h2.string)
    if string == "Adult Content Warning":
        adult_content_present = True
    else:
        adult_content_present = False
    return adult_content_present

# Retrieve Metadata for "adult" works
(i.e. fanworks rated "Mature" or "Explicit" by their author)

In [7]:
def get_rating_adult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 archive rating as string
    (value should be either 'Mature' or 'Explicit').
    """
    rating_class = soup.find_all(attrs={'class':'rating'})
    if len(rating_class) == 0:
        rating_list = "Rating not tagged"
    else:
        rating_list = []
        for item in rating_class:
            tag = item
            string = tag.string
            rating_list.append(string)
        rating_list = list(set(rating_list))
        for i in range(len(rating_list)):
            rating_list[i] = str(rating_list[i])
        if len(rating_list) == 1:
            rating_list = rating_list[0]
        else:
            pass
    return rating_list

In [8]:
def get_warnings_adult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 Archive Warnings as list.
    See Read.Me for Archive Warnings definition.
    """
    warning_class = soup.find_all(attrs={'class':'warnings'})
    if len(warning_class) == 0:
        warning_list = "Warnings not tagged"
    else:
        warnings_list = []
        for item in warning_class:
            tag = item
            string = tag.string
            warnings_list.append(string)
        warnings_list = list(set(warnings_list))
        for i in range(len(warnings_list)):
            warnings_list[i] = str(warnings_list[i])
        if len(warnings_list) == 1:
            warnings_list = warnings_list[0]
        else:
            pass
    return warnings_list

In [9]:
def get_category_adult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 category tags as list.
    See Read.Me for category tags definition.
    """
    category_class = soup.find_all(attrs={'class':'category'})
    if len(category_class) == 0:
        category_list = "No categories tagged"
    else:
        category_list = []
        for item in category_class:
            tag = item
            string = tag.string
            category_list.append(string)
        rating_list = list(set(category_list))
        for i in range(len(category_list)):
            category_list[i] = str(category_list[i])
        if len(category_list) == 1:
            category_list = category_list[0]
        else:
            pass
    return category_list

In [10]:
def get_language_adult(soup):
    """Reads in BeautifulSoup object and returns fanwork language as string."""
    language_class = soup.find_all(attrs={'class':'language'})
    language_list = []
    for item in language_class:
        tag = item
        string = str(tag.string)
        language_list.append(string)
    language = language_list[1]
    return (language)

In [11]:
def get_fandom_adult(soup):
    """
    Reads in BeautifulSoup object and returns name of Ao3 fandom(s) as list.
    """
    fandom_class = soup.find(attrs={'class':'fandoms heading'}).find('a')
    fandom_list = []
    for item in fandom_class:
        string = str(item.string)
        fandom_list.append(string)
    fandom_list = list(set(fandom_list))
    if len(fandom_list) == 1:
        fandom_list = fandom_list[0]
    else:
        pass
    return fandom_list

In [12]:
def get_relationships_adult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 relationship tags as list.
    See Read.Me for relationship tags definition.
    """
    relationships_class = soup.find_all(attrs={'class':'relationships'})
    if len(relationships_class) == 0:
        relationships_list = 'No relationships tagged'
    else:
        relationships_list = []
        for item in relationships_class:
            string = str(item.string)
            relationships_list.append(string)
        if len(relationships_list) == 1:
            relationships_list = relationships_list[0]
        else:
            pass
    return relationships_list

In [13]:
def get_characters_adult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 character tags as list.
    See Read.Me for character tags definition.
    """
    character_class = soup.find_all(attrs={'class':'characters'})
    if len(character_class) == 0:
        character_list = 'No characters tagged'
    else:
        character_list = []
        for item in character_class:
            string = str(item.string)
            character_list.append(string)
    return character_list

In [14]:
def get_additional_adult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 additional (or "freeform") tags as list.
    See Read.Me for additional / freeform tags definition.
    """
    freeform_class = soup.find_all(attrs={'class':'freeforms'})
    if len(freeform_class) == 0:
        freeform_list = 'No additional tags'
    else:
        freeform_list = []
        for item in freeform_class:
            string = str(item.string)
            freeform_list.append(string)
    return freeform_list

In [15]:
def get_date_published_adult(soup):
    """
    Reads in BeautifulSoup object and returns original fanwork publication date.
    """
    published_class = soup.find_all(attrs={'class':'published'})
    if len(published_class) == 0:
        published_list = 'Unknown'
    else:
        published_list = []
        for item in published_class:
            string = str(item.string)
            published_list.append(string)
    return published_list

In [16]:
def get_date_updated_adult(soup):
    """
    Reads in BeautifulSoup object and returns fanwork's last date updated (may
    be equivalent to date_published if fanwork has not been updated since original publication).
    """
    updated_class = soup.find_all(attrs={'class':'datetime'})
    if len(updated_class) == 0:
        date_updated = 'Unknown'
    else:
        updated_list = []
        for item in updated_class:
            string = str(item.string)
            updated_list.append(string)
        date_updated = updated_list[0]
    return date_updated

In [17]:
def get_comment_count_adult(soup):
    """
    Reads in BeautifulSoup object and returns the number of comments the fanwork has received.
    """
    comments_class = soup.find_all(attrs={'class':'comments'})
    if len(comments_class) < 2:
        comment_count = 0
    else:
        comment_count_list = []
        for item in comments_class:
            string = str(item.string)
            comment_count_list.append(string)
        comment_count = comment_count_list[1]
    return comment_count

In [18]:
def get_bookmarks_count_adult(soup):
    """
    Reads in BeautifulSoup object and returns the number of bookmarks the fanwork has received.
    """
    bookmarks_class = soup.find_all(attrs={'class':'bookmarks'})
    if len(bookmarks_class) == 0:
        bookmarks_count = 0
    else:
        bookmarks_count_list = []
        for item in bookmarks_class:
            string = str(item.string)
            bookmarks_count_list.append(string)
        bookmarks_count = bookmarks_count_list[1]
    return bookmarks_count

In [19]:
def get_hits_count_adult(soup):
    """
    Reads in BeautifulSoup object and returns the number of hits (i.e. webpage visits) the fanwork has received.
    """
    hits_class = soup.find_all(attrs={'class':'hits'})
    if len(hits_class) == 0:
        hits_count = 0
    else:
        hits_count_list = []
        for item in hits_class:
            string = str(item.string)
            hits_count_list.append(string)
        hits_count = hits_count_list[1]
    return hits_count

# Retrieve metadata for "non-adult" works
(i.e. fanworks rated "General Audiences", "Teen And Up Audiences", or "Not Rated")

In [20]:
def get_rating_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 archive rating as string
    (value should be 'General Audiences', 'Teen And Up Audiences', or 'Not Rated').
    """
    rating_class = soup.find_all(attrs={'class':'rating tags'})
    if len(rating_class) == 0:
        rating = 'Unknown'
    else:
        rating_class1 = rating_class[1]
        tag = rating_class1.find('a')
        string = str(tag.string)
        rating = string
    return rating

In [21]:
def get_warnings_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 Archive Warnings as list.
    See Read.Me for Archive Warnings definition.
    """
    warnings_list = []
    warning_class = soup.find_all(attrs={'class':'warning tags'})
    if len(warning_class) == 0:
        warnings_list = "Warnings not tagged"
    else:
        warning_class1 = warning_class[1]
        tag = warning_class1.find('a')
        string = str(tag.string)
        warnings_list.append(string)
        if len(warnings_list) == 1:
            warnings_list = warnings_list[0]
    return warnings_list

In [22]:
def get_category_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 category tags as list.
    See Read.Me for category tags definition.
    """
    category_class = soup.find_all(attrs={'class':'category tags'})
    if len(category_class) == 0:
        category = "No categories tagged"
    else:
        category_class1 = category_class[1]
        tag = category_class1.find('a')
        string = str(tag.string)
        category = string
    return category

In [65]:
def get_language_nonadult(soup):
    """Reads in BeautifulSoup object and returns fanwork language as string."""
    language_class = soup.find_all(attrs={'class':'language'})
    language_list = []
    for item in language_class:
        tag = item
        string = str(tag.string)
        language_list.append(string)
    language_list1 = language_list[1]
    language_list1 = language_list1.split('\n')
    language = language_list1[1]
    language = language.strip()
    return (language)

In [24]:
def get_fandom_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns name of Ao3 fandom(s) as list.
    """
    fandom_class = soup.find_all(attrs={'class':'fandom tags'})
    fandom_class1 = fandom_class[1]
    fandom_list = []
    for child in fandom_class1:
        tag = fandom_class1.find('a')
        string = tag.string
        fandom_list.append(string)
    fandom_list = list(set(fandom_list))
    if len(fandom_list) == 1:
        fandom = fandom_list[0]
        return fandom
    else:
        return fandom_list

In [25]:
def get_relationships_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 relationship tags as list.
    See Read.Me for relationship tags definition.
    """
    relationships_class = soup.find_all(attrs={'class':'relationship tags'})
    if len(relationships_class) == 0:
        relationships_list = "No relationships tagged"
    else:
        relationships_class1 = relationships_class[1]
        for child in relationships_class1:
            tag_list = relationships_class1.find_all('a')
            relationships_list = []
            for tag in tag_list:
                string = str(tag.string)
                relationships_list.append(string)
            if len(relationships_list) == 1:
                relationships_list = relationships_list[0]
            else:
                pass
    return relationships_list

In [26]:
def get_characters_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 character tags as list.
    See Read.Me for character tags definition.
    """
    character_class = soup.find_all(attrs={'class':'character tags'})
    if len(character_class) == 0:
        character_list = "No characters tagged"
    else:
        character_class1 = character_class[1]
        for child in character_class1:
            tag_list = character_class1.find_all('a')
            character_list = []
            for tag in tag_list:
                string = str(tag.string)
                character_list.append(string)
    return character_list

In [27]:
def get_additional_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns Ao3 additional (or "freeform") tags as list.
    See Read.Me for additional / freeform tags definition.
    """
    freeform_class = soup.find_all(attrs={'class':'freeform tags'})
    if len(freeform_class) == 0:
        freeform_list = "No additional tags"
    else:
        freeform_class1 = freeform_class[1]
        for child in freeform_class1:
            tag_list = freeform_class1.find_all('a')
            freeform_list = []
            for tag in tag_list:
                string = str(tag.string)
                freeform_list.append(string)
    return freeform_list

In [28]:
def get_date_published_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns original fanwork publication date.
    """
    published_class = soup.find_all(attrs={'class':'published'})
    if len(published_class) == 0:
        published_list = 'Unknown'
    else:
        published_list = []
        for item in published_class:
            string = str(item.string)
            published_list.append(string)
        published_list = published_list[1]
    return published_list

In [29]:
def get_date_updated_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns fanwork's last date updated (may
    be equivalent to date_published if fanwork has not been updated since original publication).
    """
    updated_class = soup.find_all(attrs={'class':'status'})
    if len(updated_class) == 0:
        updated_list = 'Unknown'
    else:
        updated_list = []
        for item in updated_class:
            string = str(item.string)
            updated_list.append(string)
        updated_list = updated_list[1]
    return updated_list

In [30]:
def get_comment_count_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns the number of comments the fanwork has received.
    """
    comments_class = soup.find_all(attrs={'class':'comments'})
    if len(comments_class) < 2:
        comments_count = 0
    else:
        string = str(comments_class[2].string)
        comments_count = int(string)
    return comments_count

In [31]:
def get_bookmarks_count_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns the number of bookmarks the fanwork has received.
    """
    bookmarks_class = soup.find_all(attrs={'class':'bookmarks'})
    if len(bookmarks_class) == 0:
        bookmarks_count = 0
    else:
        string = str(bookmarks_class[1].string)
        bookmarks_count = int(string)
    return bookmarks_count

In [32]:
def get_hits_count_nonadult(soup):
    """
    Reads in BeautifulSoup object and returns the number of hits (i.e. webpage visits) the fanwork has received.
    """
    hits_class = soup.find_all(attrs={'class':'hits'})
    if len(hits_class) == 0:
        hits_count = 0
    else:
        string = str(hits_class[1].string)
        string = string.replace(",","")
        hits_count = int(string)
    return hits_count

# Retrieve remaining metadata
(includes functions that work for both "adult" and "non-adult" works)

In [33]:
def get_word_count(soup):
    """
    Reads in BeautifulSoup object and returns the fanwork's word count.
    """
    word_count_class = soup.find_all(attrs={'class':'words'})
    word_count_list = []
    for item in word_count_class:
        string = str(item.string)
        word_count_list.append(string)
    word_count = word_count_list[1]
    word_count = word_count.replace(",","")
    word_count = int(word_count)
    return word_count

In [34]:
def get_chapter_count(soup):
    """
    Reads in BeautifulSoup object and returns the fanwork's chapter count.
    """
    chapter_count_class = soup.find_all(attrs={'class':'chapters'})
    chapter_count_list = []
    for item in chapter_count_class:
        string = str(item.string)
        chapter_count_list.append(string)
    chapter_count = chapter_count_list[1]
    return chapter_count

In [35]:
def get_kudos_count(soup):
    """
    Reads in BeautifulSoup object and returns the fanwork's kudos count.
    """
    kudos_class = soup.find_all(attrs={'class':'kudos'})
    if len(kudos_class) == 0:
        kudos_count = 0
    else:
        kudos_count_list = []
        for item in kudos_class:
            string = str(item.string)
            kudos_count_list.append(string)
        kudos_count = kudos_count_list[1]
    return kudos_count

# Create Dictionary from html items

In [36]:
def create_fanwork_dict_adult(soup):
    """
    Creates an individual fanwork dictionary - for 'adult' fanworks (i.e.
    fanworks rated 'Mature' or 'Explicit.'
    """
    fanwork_dict = {}

    # add rating
    fanwork_dict['Rating'] = get_rating_adult(soup)

    # add archive warning(s)
    fanwork_dict['Archive Warnings'] = get_warnings_adult(soup)

    # category
    fanwork_dict['Category'] = get_category_adult(soup)

    # add fandom
    fanwork_dict['Fandom'] = get_fandom_adult(soup)

    # add characters
    fanwork_dict['Characters'] = get_characters_adult(soup)

    # add relationships
    fanwork_dict['Relationships'] = get_relationships_adult(soup)

    # add additional tags
    fanwork_dict['Additional Tags'] = get_additional_adult(soup)

    # add language
    fanwork_dict['Language'] = get_language_adult(soup)

    # add stats
    fanwork_dict['Date Published'] = get_date_published_adult(soup)
    fanwork_dict['Date Updated'] = get_date_updated_adult(soup)
    fanwork_dict['Word Count'] = get_word_count(soup)
    fanwork_dict['Chapter Count'] = get_chapter_count(soup)
    fanwork_dict['Comment Count'] = get_comment_count_adult(soup)
    fanwork_dict['Kudos Count'] = get_kudos_count(soup)
    fanwork_dict['Bookmarks Count'] = get_bookmarks_count_adult(soup)
    fanwork_dict['Hits Count'] = get_hits_count_adult(soup)

    return fanwork_dict

In [37]:
def create_fanwork_dict_nonadult(soup):
    """
    Creates an individual fanwork dictionary - for 'non-adult' fanworks (i.e.
    fanworks rated 'General Audiences,' 'Teen And Up Audiences', or 'Not Rated.'
    """
    fanwork_dict = {}

    # add rating
    fanwork_dict['Rating'] = get_rating_nonadult(soup)

    # add archive warning(s)
    fanwork_dict['Archive Warnings'] = get_warnings_nonadult(soup)

    # category
    fanwork_dict['Category'] = get_category_nonadult(soup)

    # add fandom
    fanwork_dict['Fandom'] = get_fandom_nonadult(soup)

    # add characters
    fanwork_dict['Characters'] = get_characters_nonadult(soup)

    # add relationships
    fanwork_dict['Relationships'] = get_relationships_nonadult(soup)

    # add additional tags
    fanwork_dict['Additional Tags'] = get_additional_nonadult(soup)

    # add language
    fanwork_dict['Language'] = get_language_nonadult(soup)

    # add stats
    fanwork_dict['Date Published'] = get_date_published_nonadult(soup)
    fanwork_dict['Date Updated'] = get_date_updated_nonadult(soup)
    fanwork_dict['Word Count'] = get_word_count(soup)
    fanwork_dict['Chapter Count'] = get_chapter_count(soup)
    fanwork_dict['Comment Count'] = get_comment_count_nonadult(soup)
    fanwork_dict['Kudos Count'] = get_kudos_count(soup)
    fanwork_dict['Bookmarks Count'] = get_bookmarks_count_nonadult(soup)
    fanwork_dict['Hits Count'] = get_hits_count_nonadult(soup)

    return fanwork_dict

# Add fanworks_dict to master dictionary (all_works_dict)

In [38]:
def create_all_works_dict():
    """
    Creates an empty "all fanworks" dictionary.
    Individual fanwork dictionaries will be appended to this dictionary using append_all_works_dict.
    """
    dict_all_works = {}
    return dict_all_works

In [39]:
def append_all_works_dict(dict_fanwork, dict_all_works):
    """
    Appends individual fanwork dictionaries to the empty "all works" dictionary, created
    using create_all_works_dict.
    """
    current_length = len(dict_all_works)
    new_idx = current_length + 1
    dict_all_works[new_idx] = dict_fanwork
    return dict_all_works

# Convert master dictionary to json object

In [40]:
def convert_to_json(dict_all_works):
    """
    Converts nested "all works" dictionary (see create_all_works_dict and append_all_works_dict)
    to a nested JSON string.
    """
    json_string = json.dumps(dict_all_works, indent=4)
    return json_string

In [41]:
def save_json_to_file(filepath, json_string):
    """
    Takes JSON string created with convert_to_json and saves JSON file.
    """
    with open(filepath, "w") as f:
        f.write(json_string)

# Test: Individual Fanwork Links

In [42]:
# test get_fanwork_links
fanwork_links = get_fanwork_links('../data/fanwork_links.txt')
print(len(fanwork_links))

211


## Test link 1 - has adult content

In [854]:
test_url = 'https://archiveofourown.org/works/73925776'
outer_html = get_fanwork_html(test_url)
soup1 = get_soup(outer_html)
one_fanwork_dict = create_fanwork_dict_adult(soup1)
all_works_one = create_all_works_dict()
append_all_works_dict(one_fanwork_dict, all_works_one)
one_fanwork_json = convert_to_json(all_works_one)
filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/json_tests/one_fanwork_test.json'
save_json_to_file(filepath, one_fanwork_json)

TOS agreement section is present:  True
Adult content warning is present:  True


In [705]:
# Retrieve metadata
test_get_rating_adult = get_rating_adult(soup1)
print("Archive rating:")
print(test_get_rating_adult)

test_get_warnings_adult = get_warnings_adult(soup1)
print("Archive warning(s):")
print(test_get_warnings_adult)

test_get_category_adult = get_category_adult(soup1)
print("Archive categories:")
print(test_get_category_adult)

test_language1 = get_language_adult(soup1)
print("Language:")
print(test_language1)

test_fandom1 = get_fandom_adult(soup1)
print("Fandom(s):")
print(test_fandom1)

test_relationships1 = get_relationships_adult(soup1)
print("Relationship tags:")
print(test_relationships1)

test_characters1 = get_characters_adult(soup1)
print("Character tags:")
print(test_characters1)

test_additional1 = get_additional_adult(soup1)
print("Additional / freeform tags:")
print(test_additional1)

test_date_published1 = get_date_published_adult(soup1)
print("Date published:")
print(test_date_published1)

test_date_updated1 = get_date_updated_nonadult(soup1)
print("Date last updated:")
print(test_date_updated1)

test_word_count1 = get_word_count(soup1)
print("Word count:")
print(test_word_count1)

test_chapter_count1 = get_chapter_count(soup1)
print("Chapter count:")
print(test_chapter_count1)

test_comment_count1 = get_comment_count_adult(soup1)
print("Comments count:")
print(test_comment_count1)

test_kudos_count1 = get_kudos_count(soup1)
print("Kudos count:")
print(test_kudos_count1)

test_bookmarks1 = get_bookmarks_count_adult(soup1)
print("Bookmarks count:")
print(test_bookmarks1)

Explicit


## Test link 2 - does not have adult content

In [43]:
# test get_html_doc_fanworks
test_url = 'https://archiveofourown.org/works/51518734'
outer_html = get_fanwork_html(test_url)
soup2 = get_soup(outer_html)
one_fanwork_dict = create_fanwork_dict_nonadult(soup2)
all_works_one = create_all_works_dict()
append_all_works_dict(one_fanwork_dict, all_works_one)
one_fanwork_json = convert_to_json(all_works_one)
filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/json_tests/one_fanwork_test1.json'
save_json_to_file(filepath, one_fanwork_json)

TOS agreement section is present:  True
Adult content warning is present:  False


In [68]:
# save outer_html to text file for documentation purposes
outer_html = get_fanwork_html(test_url)
file_path = '/documentation_assignments/updated_documentation/samples_improved_vs_orig_info_structure/original_structure.txt'
with open(file_path, 'w') as file:
    file.write(outer_html)

TOS agreement section is present:  True
Adult content warning is present:  False


In [45]:
# Retrieve metadata
test_get_rating_nonadult = get_rating_nonadult(soup2)
print("Archive rating:")
print(test_get_rating_nonadult) # 'Teen And Up Audiences'

test_get_warnings_nonadult = get_warnings_nonadult(soup2)
print("Archive warning(s):")
print(test_get_warnings_nonadult) # 'Graphic Depictions of Violence'

test_get_category_nonadult = get_category_nonadult(soup2)
print("Archive category tags:")
print(test_get_category_nonadult) # 'F/M'

test_language2 = get_language_nonadult(soup2)
print("Language:")
print(test_language2) # 'English'

test_fandom2 = get_fandom_nonadult(soup2)
print("Fandom(s):")
print(test_fandom2) # 'Candela Obscura (Critical Role Web Series)'

test_relationships2 = get_relationships_nonadult(soup2)
print("Relationship tags:")
print(test_relationships2) # 'Jinnah "Jean" Basar/Marion Collodi'

test_characters2 = get_characters_nonadult(soup2)
print("Character tags:")
print(test_characters2) # ['Jinnah "Jean" Basar', 'Marion Collodi', 'Beatrix Monroe | Auntie Bee']

test_additional2 = get_additional_nonadult(soup2)
print("Additional / freeform tags:")
print(test_additional2) # ["Sean is also very present but he's not like. conscious", "there's ot3 energy here but this fic is About marion and jean", 'Pre-Canon', 'Surgery', 'Cuddles', 'get loved idiot']

test_date_published2 = get_date_published_nonadult(soup2)
print("Date published:")
print(test_date_published2) # '2023-11-11'

test_date_updated2 = get_date_updated_nonadult(soup2)
print("Date last updated:")
print(test_date_updated2) # '2026-04-26'

test_word_count2 = get_word_count(soup2)
print("Word count:")
print(test_word_count2) # 2779

test_chapter_count2 = get_chapter_count(soup2)
print("Chapter count:")
print(test_chapter_count2) # '2/2'

test_comment_count2 = get_comment_count_nonadult(soup2)
print("Comments count:")
print(test_comment_count2) # '15'

test_kudos_count2 = get_kudos_count(soup2)
print("Kudos count:")
print(test_kudos_count2) # '52'

test_bookmarks2 = get_bookmarks_count_adult(soup2)
print("Bookmarks count:")
print(test_bookmarks2)

Archive rating:
Teen And Up Audiences
Archive warning(s):
Graphic Depictions Of Violence
Archive category tags:
F/M
Language:

        English
      
Fandom(s):
Candela Obscura (Critical Role Web Series)
Relationship tags:
Jinnah "Jean" Basar/Marion Collodi
Character tags:
['Jinnah "Jean" Basar', 'Marion Collodi', 'Beatrix Monroe | Auntie Bee']
Additional / freeform tags:
["Sean is also very present but he's not like. conscious", "there's ot3 energy here but this fic is About marion and jean", 'Pre-Canon', 'Surgery', 'Cuddles', 'get loved idiot']
Date published:
2023-11-11
Date last updated:
2026-04-26
Word count:
2779
Chapter count:
2/2
Comments count:
15
Kudos count:
52
Bookmarks count:
8


In [46]:
print(test_language2)


        English
      


In [48]:
language_class = soup2.find_all(attrs={'class':'language'})
print(language_class)

[<dt class="language">
        Language:
      </dt>, <dd class="language" lang="en">
        English
      </dd>]


In [49]:
language_list = []
for item in language_class:
    tag = item
    string = str(tag.string)
    language_list.append(string)
print(language_list)

['\n        Language:\n      ', '\n        English\n      ']


In [52]:
print(len(language_list))

2


In [50]:
print(language_list[0])


        Language:
      


In [51]:
print(language_list[1])


        English
      


In [55]:
language_list1 = language_list[1]
language_list1 = language_list1.split('\n')
print(language_list1)
print(type(language_list1))

['', '        English', '      ']
<class 'list'>


In [57]:
print(len(language_list1))

3


In [56]:
print(language_list1[0])

In [59]:
print(language_list1[1])
print(type(language_list1[1]))

        English
<class 'str'>


In [62]:
language = language_list1[1]
language = language.strip()
print(language)
print(type(language))

English
<class 'str'>


In [63]:
# TODO - delete once you've figured out the deal with language
def get_language_nonadult(soup):
    """Reads in BeautifulSoup object and returns fanwork language as string."""
    language_class = soup.find_all(attrs={'class':'language'})
    language_list = []
    for item in language_class:
        tag = item
        string = str(tag.string)
        language_list.append(string)
    language_list1 = language_list[1]
    language_list1 = language_list1.split('\n')
    language = language_list1[1]
    language = language.strip()
    return (language)

In [64]:
get_language_nonadult(soup2)

'English'

# Test: First 10 fanworks

In [430]:
# get list of first 10 fanworks in fanwork_links list
first_10_fanworks = fanwork_links[0:10]

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606']


In [873]:
# test: first 10 links
total_fanworks = len(first_10_fanworks)
first10works_dict = create_all_works_dict()
for i in range(len(first_10_fanworks)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(first_10_fanworks[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, first10works_dict)
    print("Successfully appended dictionary to first10works_dict for link: ", str(i+1), " of ", str(total_fanworks))

Current fanwork link:  1 of 10
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  1  of  10
Successfully created soup for link:  1  of  10
Adult content is present:  None
Successfully created dictionary for link:  1  of  10
Successfully appended dictionary to first10works_dict for link:  1  of  10
Current fanwork link:  2 of 10
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  2  of  10
Successfully created soup for link:  2  of  10
Adult content is present:  None
Successfully created dictionary for link:  2  of  10
Successfully appended dictionary to first10works_dict for link:  2  of  10
Current fanwork link:  3 of 10
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  3  of  10
Successfully created soup for link:  3  of  10
Adult content is present:  None
Successfully c

In [875]:
print(len(first10works_dict)) # 10

10


In [876]:
# convert to json and save to file
first10works_json = convert_to_json(first10works_dict)
filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/first10works_test1.json'
save_json_to_file(filepath, first10works_json)

# Test: First 20 fanworks

In [44]:
first_20_fanworks = fanwork_links[0:20]

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606', 'https://archiveofourown.org/works/69266581', 'https://archiveofourown.org/works/68678736', 'https://archiveofourown.org/works/59798779', 'https://archiveofourown.org/works/55118308', 'https://archiveofourown.org/works/66001534', 'https://archiveofourown.org/works/62477026', 'https://archiveofourown.org/works/62036794', 'https://archiveofourown.org/works/60847894', 'https://archiveofourown.org/works/51300544', 'https://archiveofourown.org/works/56817901']


In [45]:
# test: first 20 links
total_fanworks = len(first_20_fanworks)
first20works_dict = create_all_works_dict()
for i in range(len(first_20_fanworks)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(first_20_fanworks[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, first20works_dict)
    print("Successfully appended dictionary to first10works_dict for link: ", str(i+1), " of ", str(total_fanworks))

Current fanwork link:  1 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  1  of  20
Successfully created soup for link:  1  of  20
Adult content is present:  True
Successfully created dictionary for link:  1  of  20
Successfully appended dictionary to first10works_dict for link:  1  of  20
Current fanwork link:  2 of 20
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  2  of  20
Successfully created soup for link:  2  of  20
Adult content is present:  False
Successfully created dictionary for link:  2  of  20
Successfully appended dictionary to first10works_dict for link:  2  of  20
Current fanwork link:  3 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  3  of  20
Successfully created soup for link:  3  of  20
Adult content is present:  True
Successfully 

In [47]:
## convert to json and save to file
first20works_json = convert_to_json(first20works_dict)
filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/first20works1.json'
save_json_to_file(filepath, first20works_json)

# Workspace (scraping from individual pagination links)
Note: I had some initial trouble with this code and memory / RAM usage. I ultimately was able to scrape each pagination link one at a time in order to get individual JSON files to later merge. This code is messier than the code above; the src and test .py scripts more cleanly represent how the code functions.

In [43]:
def get_links_to_scrape(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    links_to_scrape = open(filepath).readlines()
    return links_to_scrape

In [44]:
# test get_links_to_scrape
links_to_scrape = get_links_to_scrape('/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/pagination_links.txt')

['https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=2\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=4\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=5\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=6\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=7\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=8\n', 'https://archiveofourown.org/tags/Cande

In [46]:
## get pagination links to scrape
pagination_links1 = get_fanwork_links('/data/txt_files/fanwork_links1.txt')
pagination_links2 = get_fanwork_links('/data/txt_files/fanwork_links2.txt')
pagination_links3 = get_fanwork_links('/data/txt_files/fanwork_links3.txt')
pagination_links4 = get_fanwork_links('/data/txt_files/fanwork_links4.txt')
pagination_links5 = get_fanwork_links('/data/txt_files/fanwork_links5.txt')
pagination_links6 = get_fanwork_links('/data/txt_files/fanwork_links6.txt')
pagination_links7 = get_fanwork_links('/data/txt_files/fanwork_links7.txt')
pagination_links8 = get_fanwork_links('/data/txt_files/fanwork_links8.txt')
pagination_links9 = get_fanwork_links('/data/txt_files/fanwork_links9.txt')
pagination_links10 = get_fanwork_links('/data/txt_files/fanwork_links10.txt')
pagination_links11 = get_fanwork_links('/data/txt_files/fanwork_links11.txt')

In [48]:
pagination_links = [pagination_links1, pagination_links2, pagination_links3, pagination_links4,
                    pagination_links5, pagination_links6, pagination_links7, pagination_links8,
                    pagination_links9, pagination_links10, pagination_links11]

In [51]:
## test: all candela fics from individual pagination links (i.e. in batches)

all_candela_works_dict1 = {}
for i in range(len(pagination_links)):
    total_fanworks = len(pagination_links)
    current_page_dict = {}
    current_pagination_link = pagination_links[i]
    print("Current pagination link: " + str(i+1) + " of", str(len(pagination_links)))
    for index in range(len(current_pagination_link)):
        total_fanworks_current_page = len(current_pagination_link)
        print("Current fanwork link: " + str(index+1), " of", total_fanworks_current_page)

        ## load fanworks and create soup objects
        current_work_html = get_fanwork_html(current_pagination_link[index])
        print("Successfully created outer html for link: ", str(index+1), " of ", str(total_fanworks_current_page))

        soup = get_soup(current_work_html)
        print("Successfully created soup for link: ", str(index+1), " of ", str(total_fanworks_current_page))

        adult_content_present = detect_adult_content_warning(soup)
        print("Adult content is present: ", str(adult_content_present))

        if adult_content_present:
            current_work_dict = create_fanwork_dict_adult(soup)
        else:
            current_work_dict = create_fanwork_dict_nonadult(soup)
        print("Successfully created dictionary for link: ", str(index+1), " of ", str(total_fanworks_current_page))

        append_all_works_dict(current_work_dict, current_page_dict)
        print("Successfully appended individual fanwork dictionary", str(index+1), "to current page dict.")

    ## convert current dict to json and save to file
    current_json = convert_to_json(current_page_dict)
    json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page' + str(i+1) + '.json'
    with open(json_filepath, "w") as f:
        f.write(current_json)

Current pagination link: 1 of 1
Current fanwork link: 1  of 23
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  1  of  23
Successfully created soup for link:  1  of  23
Adult content is present:  False
Successfully created dictionary for link:  1  of  23
Successfully appended individual fanwork dictionary 1 to current page dict.
Current fanwork link: 2  of 23
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  2  of  23
Successfully created soup for link:  2  of  23
Adult content is present:  False
Successfully created dictionary for link:  2  of  23
Successfully appended individual fanwork dictionary 2 to current page dict.
Current fanwork link: 3  of 23
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  3  of  23
Successfully created soup for link:  3  of  23
Adult con

### *here is where I started scraping one pagination link at a time

In [ ]:
## one pagination link at a time
total_fanworks = len(pagination_links4)
page4works_dict = create_all_works_dict()
for i in range(len(pagination_links4)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(pagination_links4[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, page4works_dict)
    print("Successfully appended dictionary to page4works_dict for link: ", str(i+1), " of ", str(total_fanworks))

Current fanwork link:  1 of 23


In [58]:
## convert current dict to json and save to file
page4_json = convert_to_json(page4works_dict)
json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page4.json'
with open(json_filepath, "w") as f:
    f.write(current_json)

In [119]:
## one pagination link at a time
total_fanworks = len(pagination_links5)
page5works_dict = create_all_works_dict()
for i in range(len(pagination_links5)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(pagination_links5[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, page5works_dict)
    print("Successfully appended dictionary to page5works_dict for link: ", str(i+1), " of ", str(total_fanworks))

Current fanwork link:  1 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  1  of  20
Successfully created soup for link:  1  of  20
Adult content is present:  True
Successfully created dictionary for link:  1  of  20
Successfully appended dictionary to page5works_dict for link:  1  of  20
Current fanwork link:  2 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  2  of  20
Successfully created soup for link:  2  of  20
Adult content is present:  True
Successfully created dictionary for link:  2  of  20
Successfully appended dictionary to page5works_dict for link:  2  of  20
Current fanwork link:  3 of 20
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  3  of  20
Successfully created soup for link:  3  of  20
Adult content is present:  False
Successfully crea

In [56]:
# remove chapters links from each pagination link
for i in range(len(pagination_links5)):
    current_link = pagination_links5[i]
    if 'chapters' in current_link:
        pagination_links5.remove(current_link)
    else:
        continue

In [60]:
## convert current dict to json and save to file
page5_json = convert_to_json(page5works_dict)
json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page5.json'
with open(json_filepath, "w") as f:
    f.write(current_json)

In [57]:
# remove chapters links from each pagination link
for i in range(len(pagination_links6)):
    current_link = pagination_links6[i]
    if 'chapters' in current_link:
        pagination_links6.remove(current_link)
    else:
        continue

In [59]:
# remove chapters links from each pagination link
for i in range(len(pagination_links7)):
    current_link = pagination_links7[i]
    if 'chapters' in current_link:
        pagination_links7.remove(current_link)
    else:
        continue

In [98]:
## one pagination link at a time
total_fanworks = len(pagination_links7)
page7works_dict = create_all_works_dict()
for i in range(len(pagination_links7)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(pagination_links7[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, page7works_dict)
    print("Successfully appended dictionary to page7works_dict for link: ", str(i+1), " of ", str(total_fanworks))

## convert current dict to json and save to file
page7_json = convert_to_json(page7works_dict)
json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page7.json'
with open(json_filepath, "w") as f:
    f.write(current_json)

Current fanwork link:  1 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  1  of  20
Successfully created soup for link:  1  of  20
Adult content is present:  True
Successfully created dictionary for link:  1  of  20
Successfully appended dictionary to page7works_dict for link:  1  of  20
Current fanwork link:  2 of 20
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  2  of  20
Successfully created soup for link:  2  of  20
Adult content is present:  False
Successfully created dictionary for link:  2  of  20
Successfully appended dictionary to page7works_dict for link:  2  of  20
Current fanwork link:  3 of 20
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  3  of  20
Successfully created soup for link:  3  of  20
Adult content is present:  False
Successfully cr

In [61]:
# remove chapters links from each pagination link
for i in range(len(pagination_links8)):
    current_link = pagination_links8[i]
    if 'chapters' in current_link:
        pagination_links8.remove(current_link)
    else:
        continue

In [101]:
## one pagination link at a time
total_fanworks = len(pagination_links8)
page8works_dict = create_all_works_dict()
for i in range(len(pagination_links8)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(pagination_links8[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, page8works_dict)
    print("Successfully appended dictionary to page8works_dict for link: ", str(i+1), " of ", str(total_fanworks))

## convert current dict to json and save to file
page8_json = convert_to_json(page8works_dict)
json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page8.json'
with open(json_filepath, "w") as f:
    f.write(page8_json)

Current fanwork link:  1 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  1  of  20
Successfully created soup for link:  1  of  20
Adult content is present:  True
Successfully created dictionary for link:  1  of  20
Successfully appended dictionary to page8works_dict for link:  1  of  20
Current fanwork link:  2 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  2  of  20
Successfully created soup for link:  2  of  20
Adult content is present:  True
Successfully created dictionary for link:  2  of  20
Successfully appended dictionary to page8works_dict for link:  2  of  20
Current fanwork link:  3 of 20
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  3  of  20
Successfully created soup for link:  3  of  20
Adult content is present:  False
Successfully crea

In [63]:
# remove chapters links from each pagination link
for i in range(len(pagination_links9)):
    current_link = pagination_links9[i]
    if 'chapters' in current_link:
        pagination_links9.remove(current_link)
    else:
        continue

In [110]:
## one pagination link at a time
total_fanworks = len(pagination_links9)
page9works_dict = create_all_works_dict()
for i in range(len(pagination_links9)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(pagination_links9[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, page9works_dict)
    print("Successfully appended dictionary to page9works_dict for link: ", str(i+1), " of ", str(total_fanworks))

## convert current dict to json and save to file
page9_json = convert_to_json(page9works_dict)
json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page9.json'
with open(json_filepath, "w") as f:
    f.write(page9_json)

Current fanwork link:  1 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  1  of  20
Successfully created soup for link:  1  of  20
Adult content is present:  True
Successfully created dictionary for link:  1  of  20
Successfully appended dictionary to page8works_dict for link:  1  of  20
Current fanwork link:  2 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  2  of  20
Successfully created soup for link:  2  of  20
Adult content is present:  True
Successfully created dictionary for link:  2  of  20
Successfully appended dictionary to page8works_dict for link:  2  of  20
Current fanwork link:  3 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  3  of  20
Successfully created soup for link:  3  of  20
Adult content is present:  True
Successfully create

In [64]:
# remove chapters links from each pagination link
for i in range(len(pagination_links10)):
    current_link = pagination_links10[i]
    if 'chapters' in current_link:
        pagination_links10.remove(current_link)
    else:
        continue

In [113]:
## one pagination link at a time
total_fanworks = len(pagination_links10)
page10works_dict = create_all_works_dict()
for i in range(len(pagination_links10)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(pagination_links10[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, page10works_dict)
    print("Successfully appended dictionary to page10works_dict for link: ", str(i+1), " of ", str(total_fanworks))

## convert current dict to json and save to file
page10_json = convert_to_json(page10works_dict)
json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page10.json'
with open(json_filepath, "w") as f:
    f.write(page10_json)

Current fanwork link:  1 of 20
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  1  of  20
Successfully created soup for link:  1  of  20
Adult content is present:  False
Successfully created dictionary for link:  1  of  20
Successfully appended dictionary to page10works_dict for link:  1  of  20
Current fanwork link:  2 of 20
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  2  of  20
Successfully created soup for link:  2  of  20
Adult content is present:  False
Successfully created dictionary for link:  2  of  20
Successfully appended dictionary to page10works_dict for link:  2  of  20
Current fanwork link:  3 of 20
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  3  of  20
Successfully created soup for link:  3  of  20
Adult content is present:  True
Successfully 

In [66]:
# remove chapters links from each pagination link
for i in range(len(pagination_links11)):
    current_link = pagination_links11[i]
    if 'chapters' in current_link:
        pagination_links11.remove(current_link)
    else:
        continue

In [116]:
## one pagination link at a time
total_fanworks = len(pagination_links11)
page11works_dict = create_all_works_dict()
for i in range(len(pagination_links11)):
    print("Current fanwork link: ", str(i+1), "of", str(total_fanworks))

    current_work_html = get_fanwork_html(pagination_links11[i])
    print("Successfully created outer html for link: ", str(i+1), " of ", str(total_fanworks))

    soup = get_soup(current_work_html)
    print("Successfully created soup for link: ", str(i+1), " of ", str(total_fanworks))

    adult_content_present = detect_adult_content_warning(soup)
    print("Adult content is present: ", str(adult_content_present))

    if adult_content_present:
        current_work_dict = create_fanwork_dict_adult(soup)
    else:
        current_work_dict = create_fanwork_dict_nonadult(soup)
    print("Successfully created dictionary for link: ", str(i+1), " of ", str(total_fanworks))

    append_all_works_dict(current_work_dict, page11works_dict)
    print("Successfully appended dictionary to page11works_dict for link: ", str(i+1), " of ", str(total_fanworks))

## convert current dict to json and save to file
page11_json = convert_to_json(page11works_dict)
json_filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page11.json'
with open(json_filepath, "w") as f:
    f.write(page11_json)

Current fanwork link:  1 of 11
TOS agreement section is present:  True
Adult content warning is present:  False
Successfully created outer html for link:  1  of  11
Successfully created soup for link:  1  of  11
Adult content is present:  False
Successfully created dictionary for link:  1  of  11
Successfully appended dictionary to page11works_dict for link:  1  of  11
Current fanwork link:  2 of 11
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  2  of  11
Successfully created soup for link:  2  of  11
Adult content is present:  True
Successfully created dictionary for link:  2  of  11
Successfully appended dictionary to page11works_dict for link:  2  of  11
Current fanwork link:  3 of 11
TOS agreement section is present:  True
Adult content warning is present:  True
Successfully created outer html for link:  3  of  11
Successfully created soup for link:  3  of  11
Adult content is present:  True
Successfully cr

In [66]:
# testing pagination page 2 link 16: 'https://archiveofourown.org/works/56944855'
test_url = 'https://archiveofourown.org/works/56944855'
outer_html = get_fanwork_html(test_url)
soup2 = get_soup(outer_html)
# one_fanwork_dict = create_fanwork_dict_nonadult(soup2)
# all_works_one = create_all_works_dict()
# append_all_works_dict(one_fanwork_dict, all_works_one)
# one_fanwork_json = convert_to_json(all_works_one)
# filepath = '/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/one_fanwork_test2.json'
# save_json_to_file(filepath, one_fanwork_json)

TOS agreement section is present:  True
Adult content warning is present:  False


In [67]:
# code adapted from: https://www.geeksforgeeks.org/python/how-to-merge-multiple-json-files-using-python/
def merge_json_files(file_paths, output_file):
    merged_data = []
    for path in file_paths:
        with open(path, 'r') as file:
            data = json.load(file)
            merged_data.append(data)
    with open(output_file, 'w') as outfile:
        json.dump(merged_data, outfile)

In [76]:
# append individual jsons to all_candela_works_dict1
file_paths = ["/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page1.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page2.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page3.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page4.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page5.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page7.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page8.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page9.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page10.json",
              "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/candela_page11.json",
              ]

In [77]:
# create new JSON file
output_dict = {}
output_file = convert_to_json(output_dict)
output_filepath = "/home/ekmys/PycharmProjects/IMT542_FinalProject_Stelter/data/json_files/all_candela_works1.json"
with open(output_filepath, 'w') as f:
    f.write(output_file)

merge_json_files(file_paths, output_filepath)
print(f"Merged data written to '{output_file}'")

Merged data written to '{}'
